In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import nltk
import warnings
warnings.filterwarnings('ignore')

from nltk.corpus import stopwords
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

nltk.download('stopwords')

print("All libraries loaded successfully!")

In [ ]:
df = pd.read_csv('all_tickets_processed_improved_v3.csv')

print(f"Shape: {df.shape}")
print(f"Columns: {list(df.columns)}")
print(f"\nCategory distribution:")
print(df['Topic_group'].value_counts())
df.head()

In [ ]:
# Drop rows with missing values
df = df.dropna(subset=['Document', 'Topic_group'])

# Clean the text
stop_words = set(stopwords.words('english'))

def clean_text(text):
    # Lowercase
    text = text.lower()
    # Remove short words and stopwords
    words = text.split()
    words = [w for w in words if w not in stop_words and len(w) > 2]
    return ' '.join(words)

df['cleaned_text'] = df['Document'].apply(clean_text)

print(f"Total tickets after cleaning: {len(df)}")
print("\nSample cleaned text:")
print(df['cleaned_text'].iloc[0])

In [ ]:
# Assign priority based on category
# This mirrors real business logic used in IT support teams
priority_map = {
    'Hardware': 'High',
    'Access': 'High',
    'Administrative rights': 'High',
    'Storage': 'Medium',
    'Purchase': 'Medium',
    'Internal Project': 'Medium',
    'HR Support': 'Low',
    'Miscellaneous': 'Low'
}

df['Priority'] = df['Topic_group'].map(priority_map)

print("Priority distribution:")
print(df['Priority'].value_counts())

In [ ]:
plt.figure(figsize=(12, 5))
colors = ['#2196F3', '#4CAF50', '#FF9800', '#E91E63', 
          '#9C27B0', '#00BCD4', '#FF5722', '#607D8B']

df['Topic_group'].value_counts().plot(kind='bar', color=colors)
plt.title('Ticket Category Distribution', fontsize=16)
plt.xlabel('Category')
plt.ylabel('Number of Tickets')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig('category_distribution.png', dpi=150)
plt.show()
print("Chart saved!")

In [ ]:
plt.figure(figsize=(7, 5))
priority_colors = {'High': '#F44336', 'Medium': '#FF9800', 'Low': '#4CAF50'}
counts = df['Priority'].value_counts()

plt.bar(counts.index, counts.values, 
        color=[priority_colors[p] for p in counts.index])
plt.title('Ticket Priority Distribution', fontsize=16)
plt.xlabel('Priority Level')
plt.ylabel('Number of Tickets')
plt.tight_layout()
plt.savefig('priority_distribution.png', dpi=150)
plt.show()
print("Chart saved!")

In [ ]:
# Convert text to numbers using TF-IDF
# TF-IDF gives higher weight to words that are important 
# in a ticket but rare across all tickets
vectorizer = TfidfVectorizer(max_features=5000, ngram_range=(1,2))
X = vectorizer.fit_transform(df['cleaned_text'])
y = df['Topic_group']

print(f"Feature matrix shape: {X.shape}")
print(f"Each ticket is now represented as {X.shape[1]} numbers")

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Training tickets: {X_train.shape[0]}")
print(f"Testing tickets : {X_test.shape[0]}")

In [ ]:
# Train Logistic Regression model
lr_model = LogisticRegression(max_iter=1000, random_state=42)
lr_model.fit(X_train, y_train)

# Predict on test data
y_pred_lr = lr_model.predict(X_test)

# Accuracy
acc = accuracy_score(y_test, y_pred_lr)
print(f"Category Classification Accuracy: {acc*100:.2f}%")

In [ ]:
print("DETAILED CLASSIFICATION REPORT")
print("=" * 60)
print(classification_report(y_test, y_pred_lr))

In [ ]:
plt.figure(figsize=(10, 8))
cm = confusion_matrix(y_test, y_pred_lr, labels=lr_model.classes_)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=lr_model.classes_,
            yticklabels=lr_model.classes_)
plt.title('Confusion Matrix - Ticket Category Classification', fontsize=14)
plt.ylabel('Actual Category')
plt.xlabel('Predicted Category')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=150)
plt.show()
print("Chart saved!")

In [ ]:
# Now train a separate model to predict priority
y_priority = df['Priority']

X_train_p, X_test_p, y_train_p, y_test_p = train_test_split(
    X, y_priority, test_size=0.2, random_state=42, stratify=y_priority
)

priority_model = LogisticRegression(max_iter=1000, random_state=42)
priority_model.fit(X_train_p, y_train_p)

y_pred_priority = priority_model.predict(X_test_p)
acc_p = accuracy_score(y_test_p, y_pred_priority)

print(f"Priority Prediction Accuracy: {acc_p*100:.2f}%")
print("\nPriority Classification Report:")
print(classification_report(y_test_p, y_pred_priority))

In [ ]:
def classify_ticket(ticket_text):
    cleaned = clean_text(ticket_text)
    vectorized = vectorizer.transform([cleaned])
    
    category = lr_model.predict(vectorized)[0]
    priority = priority_model.predict(vectorized)[0]
    
    cat_confidence = lr_model.predict_proba(vectorized).max() * 100
    
    print("=" * 50)
    print("TICKET CLASSIFICATION RESULT")
    print("=" * 50)
    print(f"Ticket   : {ticket_text[:80]}...")
    print(f"Category : {category}")
    print(f"Priority : {priority}")
    print(f"Confidence: {cat_confidence:.1f}%")
    print("=" * 50)

# Test with sample tickets
classify_ticket("My laptop screen is broken and I cannot work at all")
print()
classify_ticket("I need access to the HR portal to submit my leave request")
print()
classify_ticket("Can you help me understand the new expense policy")

In [ ]:
print("""CUSTOMER SUPPORT TICKET CLASSIFICATION REPORT

This system reads incoming support tickets and automatically assigns each one a category and a priority level. It was
trained on 47,000 real IT support tickets across 8 categories including Hardware, Access, HR Support, Storage and more.

Working: Each ticket goes through text cleaning and is then converted into numbers using TF-IDF. A Logistic Regression model reads
those numbers and matches the ticket to the category it most closely resembles based on what it learned during training.
Priority is based on business logic tied to each category. Hardware and Access issues are High priority because they
stop people from working. Storage and Purchase requests are Medium. HR queries and general questions are Low priority
since they are less time sensitive.

Overall, this system removes the need for manual ticket sorting. The moment a ticket comes in, it gets a category and a
priority. Urgent issues surface immediately instead of getting buried under low priority requests. Support agents
spend their time solving problems rather than reading and routing tickets.

Model Training Related:
Category Classification Accuracy: see Cell 9 output
Priority Prediction Accuracy: see Cell 12 output
Training data: 80% of 47,000 tickets
Test data: 20% of 47,000 tickets

Data source: IT Service Ticket Classification Dataset, Kaggle
""")